# Phase 2 

### chargement du dataset

In [9]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Classes :", data.target_names)
print("\nRépartition de la cible :")
print(y.value_counts(normalize=True).sort_index().round(3))


Classes : ['malignant' 'benign']

Répartition de la cible :
target
0    0.373
1    0.627
Name: proportion, dtype: float64


### evaluer la stabilité du model 

In [10]:
modele = make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000, random_state=42))
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):
    

    rng = np.random.default_rng(random_state)
    n_lignes = len(X)
    indices = np.arange(n_lignes)

    scores = []

    for iteration in range(n_iterations):
        indices_bootstrap = rng.choice(
            indices,
            size=n_lignes,
            replace=True
        )

        indices_oob = indices[~np.isin(indices, indices_bootstrap)]

        if len(indices_oob) == 0:
            print(f"Itération {iteration + 1} ignorée : aucun point out-of-bag.")
            continue

        X_bootstrap = X.iloc[indices_bootstrap]
        y_bootstrap = y.iloc[indices_bootstrap]

        X_oob = X.iloc[indices_oob]
        y_oob = y.iloc[indices_oob]

        modele_iteration = clone(modele)
        modele_iteration.fit(X_bootstrap, y_bootstrap)

        score = modele_iteration.score(X_oob, y_oob)
        scores.append(score)

    if len(scores) == 0:
        raise ValueError("Aucun score n'a pu être calculé.")

    moyenne = np.mean(scores)

    if len(scores) == 1:
        print(f"Score obtenu sur 1 bootstrap : {moyenne:.3f}")
        print("Écart-type non calculable avec une seule itération.")
    else:
        ecart_type = np.std(scores, ddof=1)
        print(f"Score moyen sur {len(scores)} bootstraps : {moyenne:.3f} (± {ecart_type:.3f})")

    return scores
scores_bootstrap = bootstrap_scores(
    modele=modele,
    X=X,
    y=y,
    n_iterations=30,
    random_state=42
)

print("\nNombre de scores obtenus :", len(scores_bootstrap))
print("Premiers scores :", np.round(scores_bootstrap[:5], 3))

Score moyen sur 30 bootstraps : 0.976 (± 0.010)

Nombre de scores obtenus : 30
Premiers scores : [0.972 0.972 0.972 0.981 0.98 ]


#### cas normal

In [11]:
if len(scores_bootstrap) > 1:
    print("Checkpoint cas normal : validé.")
    print("La fonction renvoie plusieurs scores.")
    print("Moyenne :", round(np.mean(scores_bootstrap), 3))
    print("Écart-type :", round(np.std(scores_bootstrap, ddof=1), 3))
else:
    print("Checkpoint cas normal : problème, pas assez de scores.")

Checkpoint cas normal : validé.
La fonction renvoie plusieurs scores.
Moyenne : 0.976
Écart-type : 0.01


#### Tester n_iterations=1

In [12]:
scores_une_iteration = bootstrap_scores(modele=modele,X=X,y=y,n_iterations=1,random_state=42)

print("\nScores obtenus :", np.round(scores_une_iteration, 3))

Score obtenu sur 1 bootstrap : 0.972
Écart-type non calculable avec une seule itération.

Scores obtenus : [0.972]


### Comparaison avec et sans remise

#### le problème sans remise avec l’OOB

In [14]:
rng = np.random.default_rng(42)
n_lignes = len(X)
indices = np.arange(n_lignes)

indices_sans_remise = rng.choice(
    indices,
    size=n_lignes,
    replace=False
)

indices_oob_sans_remise = indices[~np.isin(indices, indices_sans_remise)]

print("Nombre de lignes du dataset :", n_lignes)
print("Nombre de lignes tirées sans remise :", len(indices_sans_remise))
print("Nombre de points out-of-bag :", len(indices_oob_sans_remise))

if len(indices_oob_sans_remise) == 0:
    print("Sans remise, aucun point out-of-bag n'est disponible.")
    print("Ce n'est donc pas un vrai bootstrap.")

Nombre de lignes du dataset : 569
Nombre de lignes tirées sans remise : 569
Nombre de points out-of-bag : 0
Sans remise, aucun point out-of-bag n'est disponible.
Ce n'est donc pas un vrai bootstrap.


#### Préparer un train/test fixe pour comparer la dispersion

In [15]:
X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train comparaison :", len(X_train_comp))
print("Test comparaison :", len(X_test_comp))

Train comparaison : 455
Test comparaison : 114


#### la fonction de comparaison avec/sans remise

In [16]:
def comparer_scores_avec_sans_remise(modele, X_train, y_train, X_test, y_test, n_iterations=30, random_state=42):
    

    rng = np.random.default_rng(random_state)
    n_lignes_train = len(X_train)
    indices_train = np.arange(n_lignes_train)

    scores_avec_remise = []
    scores_sans_remise = []

    for iteration in range(n_iterations):
        indices_avec_remise = rng.choice(
            indices_train,
            size=n_lignes_train,
            replace=True
        )

        indices_sans_remise = rng.choice(
            indices_train,
            size=n_lignes_train,
            replace=False
        )

        modele_avec = clone(modele)
        modele_avec.fit(
            X_train.iloc[indices_avec_remise],
            y_train.iloc[indices_avec_remise]
        )
        scores_avec_remise.append(modele_avec.score(X_test, y_test))

        modele_sans = clone(modele)
        modele_sans.fit(
            X_train.iloc[indices_sans_remise],
            y_train.iloc[indices_sans_remise]
        )
        scores_sans_remise.append(modele_sans.score(X_test, y_test))

    return scores_avec_remise, scores_sans_remise
scores_avec_remise, scores_sans_remise = comparer_scores_avec_sans_remise(
    modele=modele,
    X_train=X_train_comp,
    y_train=y_train_comp,
    X_test=X_test_comp,
    y_test=y_test_comp,
    n_iterations=30,
    random_state=42
)

print("Nombre de scores avec remise :", len(scores_avec_remise))
print("Nombre de scores sans remise :", len(scores_sans_remise))

print("\nÉcart-type avec remise :", round(np.std(scores_avec_remise, ddof=1), 3))
print("Écart-type sans remise :", round(np.std(scores_sans_remise, ddof=1), 3))

Nombre de scores avec remise : 30
Nombre de scores sans remise : 30

Écart-type avec remise : 0.011
Écart-type sans remise : 0.0
